In [1]:
import torch
import torch.nn as nn

In [3]:
loss_fct_hallucination = nn.CrossEntropyLoss(ignore_index=-100)
loss_fct_token = nn.CrossEntropyLoss(ignore_index=-100)

In [49]:
token_pred = torch.tensor([[10, 9, 9], [1, 22, 3], [1, 1, 1.0]])
token_labels = torch.tensor([0, -100, -100])
token_loss = loss_fct_token(token_pred, token_labels)
print(torch.nn.Softmax(dim=-1)(token_pred))
print(token_loss)

tensor([[5.7612e-01, 2.1194e-01, 2.1194e-01],
        [7.5826e-10, 1.0000e+00, 5.6028e-09],
        [3.3333e-01, 3.3333e-01, 3.3333e-01]])
tensor(0.5514)


In [47]:
hallucination_pred = torch.tensor([[11.0, 2, 3], [1, 22, 3], [1, 1, 1]])
hallucination_labels = torch.tensor([0, 1, 2])
hallucination_loss = loss_fct_hallucination(hallucination_pred, hallucination_labels)
print(hallucination_loss)

tensor(0.3664)


In [48]:
total_loss = token_loss + hallucination_loss
print(total_loss)

tensor(81.0595)


In [5]:
mask_no_hallucination = torch.tensor([True, False, False])
mask_is_deletion_token = torch.tensor([False, False, True])

In [6]:
combined_mask = (mask_no_hallucination | mask_is_deletion_token).unsqueeze(-1)

In [7]:
combined_mask

tensor([[ True],
        [False],
        [ True]])

In [35]:
# correction_weight_tensor = [1.0, 3.0]
correction_weight_tensor = [1.0, 1.0]

hallucination_logits = torch.tensor([[[0.1], [0.9]], [[-10], [0.1]], [[-10], [0.2]]])
print(hallucination_logits.shape)
hallucination_labels = torch.tensor([[0, -100], [0, 0], [1, 0]])
print(hallucination_labels.shape)

torch.Size([3, 2, 1])
torch.Size([3, 2])


In [36]:
# --- 2. Calculate Hallucination Detection Loss (Binary Cross-Entropy) ---
# We now use BCEWithLogitsLoss for the binary (0 or 1) hallucination task.
pos_weight = None
if correction_weight_tensor is not None:
    # Assumes correction_weights is [weight_for_class_0, weight_for_class_1]
    # pos_weight is the ratio of negative to positive weights.
    pos_weight = correction_weight_tensor[1] / correction_weight_tensor[0]
    pos_weight = torch.tensor([pos_weight]).to(hallucination_logits.device)
    print(pos_weight)

loss_fct_hallucination = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

shift_hallucination_logits = hallucination_logits[..., :-1, :].contiguous()
print(f"shift_hallucination_logits:\n{shift_hallucination_logits}")
shift_hallucination_labels = hallucination_labels[..., 1:].contiguous()
print(f"shift_hallucination_labels:\n{shift_hallucination_labels}")

# Reshape for BCEWithLogitsLoss
shift_hallucination_logits = shift_hallucination_logits.view(-1)
print(f"shift_hallucination_logits:\n{shift_hallucination_logits}")
shift_hallucination_labels = shift_hallucination_labels.view(-1).float()
print(f"shift_hallucination_labels:\n{shift_hallucination_labels}")

# Create a mask to ignore padding tokens (-100)
active_loss_mask = shift_hallucination_labels != -100
print(f"active_loss_mask:\n{active_loss_mask}")
# Apply the mask to get only the active logits and labels
active_logits = shift_hallucination_logits[active_loss_mask]
print(f"active_logits:\n{active_logits}")
active_labels = shift_hallucination_labels[active_loss_mask]
print(f"active_labels:\n{active_labels}")

# Calculate loss only on active elements
hallucination_loss = loss_fct_hallucination(active_logits, active_labels)

tensor([1.])
shift_hallucination_logits:
tensor([[[  0.1000]],

        [[-10.0000]],

        [[-10.0000]]])
shift_hallucination_labels:
tensor([[-100],
        [   0],
        [   0]])
shift_hallucination_logits:
tensor([  0.1000, -10.0000, -10.0000])
shift_hallucination_labels:
tensor([-100.,    0.,    0.])
active_loss_mask:
tensor([False,  True,  True])
active_logits:
tensor([-10., -10.])
active_labels:
tensor([0., 0.])


In [37]:
print(hallucination_loss)

tensor(4.5776e-05)
